# Tarefa 3: Multilayer Perceptron — Liver Disorder

**Curso:** FT108A — Introdução ao Aprendizado de Máquina, 2026S2.

Este relatório aplica redes MLP ao conjunto BUPA Liver Disorders com a mesma metodologia da Tarefa 2: cinco partições aleatórias estratificadas 70/30. Compara-se uma implementação própria em NumPy, uma rede em PyTorch e o `MLPClassifier` do scikit-learn, e os erros médios de teste com os da atividade anterior.


## 1. Ferramentas e classificadores

| Implementação | Biblioteca / versão | Observação |
| --- | --- | --- |
| MLP do zero | NumPy | forward/backward e SGD manuais; sigmoide nas duas camadas |
| MLP PyTorch | `torch` | `Linear→ReLU→Linear`, `BCEWithLogitsLoss`, Adam |
| MLP ferramenta | scikit-learn `MLPClassifier` | `activation="logistic"`, `solver="adam"` |

O enunciado pede ferramenta com MLP: o scikit-learn cobre esse requisito. NumPy e PyTorch entram para aprendizado e comparação no mesmo protocolo.


## 2. Dados, alvo e pré-processamento

Conjunto BUPA Liver Disorders (UCI / Forsyth, 1990): cinco exames (`mcv`, `alkphos`, `sgpt`, `sgot`, `gammagt`) e `drinks`. O campo `selector` não é rótulo clínico.

Alvo binário (Turney, 1995), como na Tarefa 2:

- `heavy_drinker = 1` se `drinks >= 3`;
- `heavy_drinker = 0` caso contrário.

Features: apenas os cinco exames. Em cada repetição, média e desvio são estimados **somente no treino** e aplicados ao teste (padronização z-score), o que evita vazamento e estabiliza o treino das MLPs.


## 3. Parâmetros

| Modelo | Parâmetros |
| --- | --- |
| NumPy | `H=10`, `lr=0.1`, `epochs=200`, inicialização normal (`scale=0.1`), bias zero; BCE + SGD em batch cheio |
| PyTorch | `H=10`, `lr=1e-2`, `epochs=200`, Adam, `BCEWithLogitsLoss`, ReLU na oculta |
| scikit-learn | `hidden_layer_sizes=(10,)`, `activation="logistic"`, `solver="adam"`, `max_iter=500`, `random_state=seed` |

Não houve busca sistemática de hiperparâmetros.


## 4. Protocolo experimental

- Cinco repetições com sementes `42`, `7`, `13`, `21` e `99`.
- Split estratificado 70/30 (`test_size=0.3`, `stratify=y`).
- Em cada repetição, os três modelos usam a **mesma** partição (mesma seed).
- Métrica: erro de classificação no teste (`1 −` acurácia), limiar 0,5.
- Arquivos de cada partição: `partitions/rep{k}/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn

DATA_PATH = Path("../../../data/liver.csv")
PART_DIR = Path("partitions")
SEEDS = [42, 7, 13, 21, 99]
TEST_SIZE = 0.30

df = pd.read_csv(DATA_PATH)
feature_cols = ["mcv", "alkphos", "sgpt", "sgot", "gammagt"]
X = df[feature_cols].to_numpy(dtype=float)
y = (df["drinks"].to_numpy(dtype=float) >= 3).astype(float)
print(X.shape, y.shape, "positivos=", float(y.mean()))


(345, 5) (345,) positivos= 0.5101449275362319


In [2]:
def standardize_fit(X_train):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std = np.where(std < 1e-8, 1e-8, std)
    return mean, std

def standardize_apply(X, mean, std):
    return (X - mean) / std

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def init_weights(d, H, rng):
    return {
        "W1": rng.normal(scale=0.1, size=(d, H)),
        "b1": np.zeros(H),
        "W2": rng.normal(scale=0.1, size=(H, 1)),
        "b2": np.zeros(1),
    }

def forward(X, params):
    z1 = X @ params["W1"] + params["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    y_hat = sigmoid(z2)
    return y_hat, {"X": X, "a1": a1}

def binary_cross_entropy(y_true, y_hat, eps=1e-9):
    y = y_true.reshape(-1)
    p = np.clip(y_hat.reshape(-1), eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def classification_error(y_true, y_hat, threshold=0.5):
    y = y_true.reshape(-1)
    pred = (y_hat.reshape(-1) >= threshold).astype(float)
    return float(np.mean(pred != y))

def backward(y_true, y_hat, cache, params):
    X, a1 = cache["X"], cache["a1"]
    n = X.shape[0]
    y = y_true.reshape(-1, 1)
    y_hat = y_hat.reshape(-1, 1)
    delta2 = (y_hat - y) / n
    dW2 = a1.T @ delta2
    db2 = np.sum(delta2, axis=0)
    delta1 = (delta2 @ params["W2"].T) * a1 * (1 - a1)
    dW1 = X.T @ delta1
    db1 = np.sum(delta1, axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

def sgd_step(params, grads, lr):
    for k in params:
        params[k] = params[k] - lr * grads[k]

def train_mlp_numpy(X_train, y_train, X_test, y_test, H=10, lr=0.1, epochs=200, seed=0):
    rng = np.random.default_rng(seed)
    params = init_weights(X_train.shape[1], H, rng)
    for _ in range(epochs):
        y_hat, cache = forward(X_train, params)
        grads = backward(y_train, y_hat, cache, params)
        sgd_step(params, grads, lr)
    y_te, _ = forward(X_test, params)
    return classification_error(y_test, y_te)

class MLPTorch(nn.Module):
    def __init__(self, d, H=10):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, H), nn.ReLU(), nn.Linear(H, 1))
    def forward(self, x):
        return self.net(x)

def train_mlp_torch(X_train, y_train, X_test, y_test, H=10, lr=1e-2, epochs=200, seed=0):
    torch.manual_seed(seed)
    model = MLPTorch(X_train.shape[1], H)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy().reshape(-1)
    return float(np.mean((probs >= 0.5).astype(float) != y_test.reshape(-1)))

print("funções OK")


funções OK


In [3]:
def run_all(X, y, save_partitions=True):
    err_np, err_torch, err_sk = [], [], []
    for k, seed in enumerate(SEEDS, start=1):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
        )
        mean, std = standardize_fit(X_tr)
        X_tr_s = standardize_apply(X_tr, mean, std)
        X_te_s = standardize_apply(X_te, mean, std)
        if save_partitions:
            rep = PART_DIR / f"rep{k}"
            rep.mkdir(parents=True, exist_ok=True)
            np.savetxt(rep / "X_train.csv", X_tr, delimiter=",")
            np.savetxt(rep / "X_test.csv", X_te, delimiter=",")
            np.savetxt(rep / "y_train.csv", y_tr, delimiter=",")
            np.savetxt(rep / "y_test.csv", y_te, delimiter=",")
        e_np = train_mlp_numpy(X_tr_s, y_tr, X_te_s, y_te, seed=seed)
        e_th = train_mlp_torch(X_tr_s, y_tr, X_te_s, y_te, seed=seed)
        clf = MLPClassifier(
            hidden_layer_sizes=(10,), activation="logistic", solver="adam",
            max_iter=500, random_state=seed,
        )
        clf.fit(X_tr_s, y_tr)
        e_sk = float(np.mean(clf.predict(X_te_s) != y_te))
        err_np.append(e_np); err_torch.append(e_th); err_sk.append(e_sk)
        print(f"rep{k} seed={seed}  numpy={e_np:.4f}  torch={e_th:.4f}  sklearn={e_sk:.4f}")
    return err_np, err_torch, err_sk

errors_np, errors_torch, errors_sk = run_all(X, y)
print("médias:", float(np.mean(errors_np)), float(np.mean(errors_torch)), float(np.mean(errors_sk)))


rep1 seed=42  numpy=0.4808  torch=0.4135  sklearn=0.3365


rep2 seed=7  numpy=0.4231  torch=0.4519  sklearn=0.4135
rep3 seed=13  numpy=0.5096  torch=0.3462  sklearn=0.3365


rep4 seed=21  numpy=0.4038  torch=0.4615  sklearn=0.4712


rep5 seed=99  numpy=0.4135  torch=0.4808  sklearn=0.4712
médias: 0.44615384615384607 0.43076923076923085 0.4057692307692308


## 5. Resultados

Erros de classificação médios no teste (5 repetições):

| Modelo | Erro médio (teste) |
| --- | --- |
| MLP NumPy | 0,4462 |
| MLP PyTorch | 0,4308 |
| MLP scikit-learn | 0,4058 |
| Árvore (Tarefa 2) | 0,4327 |
| Naïve Bayes (Tarefa 2) | 0,4346 |
| k-NN (Tarefa 2) | 0,4423 |
| Ensemble (Tarefa 2) | 0,4365 |

Erros por repetição:

| Rep (seed) | NumPy | PyTorch | scikit-learn |
| --- | --- | --- | --- |
| 1 (42) | 0,4808 | 0,4135 | 0,3365 |
| 2 (7) | 0,4231 | 0,4519 | 0,4135 |
| 3 (13) | 0,5096 | 0,3462 | 0,3365 |
| 4 (21) | 0,4038 | 0,4615 | 0,4712 |
| 5 (99) | 0,4135 | 0,4808 | 0,4712 |


## 6. Discussão

O `MLPClassifier` obteve o menor erro médio (0,4058) entre as três MLPs e ficou abaixo dos erros da Tarefa 2 (cerca de 0,43–0,44). A implementação NumPy (sigmoide + SGD em batch) ficou em 0,4462, um pouco acima desse patamar, o que é compatível com otimização mais simples e hiperparâmetros fixos. O PyTorch (ReLU + Adam) situou-se no meio (0,4308).

O Liver com alvo Turney é um problema ruidoso e de tamanho modesto (345 instâncias): diferenças de poucos pontos percentuais entre modelos são esperadas e não autorizam conclusões fortes sem validação adicional. Ainda assim, no protocolo fixado, a MLP da ferramenta (scikit-learn) foi a mais competitiva.

Ajustes documentados nos dados: definição do alvo, exclusão de `selector`/`drinks` das features e padronização por partição.


## 7. Entregáveis

- PDF deste relatório.
- ZIP com o PDF e os arquivos CSV de cada partição (`rep1`…`rep5`).

*Conteúdo, experimentos e conclusões são meus; a formatação do texto teve assistência de IA.*
